In [1]:
!pip -q install --no-cache-dir "numpy<2" "pandas"

!pip -q install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121

!pip -q install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
  -f https://data.pyg.org/whl/torch-2.2.2+cu121.html

!pip -q install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 196.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but yo

In [1]:
import numpy as np, torch
import pandas as pd
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("pandas:", pd.__version__)

numpy: 1.26.4
torch: 2.2.2+cu121
pandas: 2.2.2


In [2]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
base_path = "/content/drive/MyDrive/dataset_cleaned/HI-Medium_Trans.csv"  # change if needed

df = pd.read_csv(base_path)
print("Shape:", df.shape)
df.head()

Shape: (31898218, 21)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,...,8.824035,2022-09-01 00:17:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,...,8.954194,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,...,7.539681,2022-09-01 00:17:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,...,18.119128,2022-09-01 00:03:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,...,17.641288,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0


In [5]:
required_cols = [
    "From Bank", "Account", "To Bank", "Account.1",
    "Is Laundering",
    "Log Amount Received", "_ts",
    "tx_dow", "tx_is_weekend",
    "tx_hour_sin", "tx_hour_cos"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

In [6]:
# Build nodes
src_key = df["From Bank"].astype(str) + "|" + df["Account"].astype(str)
dst_key = df["To Bank"].astype(str) + "|" + df["Account.1"].astype(str)

all_keys = pd.concat([src_key, dst_key], ignore_index=True)
codes, uniques = pd.factorize(all_keys, sort=False)

E = len(src_key)
src = codes[:E].astype("int64")
dst = codes[E:].astype("int64")
num_nodes = len(uniques)

edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)

print("num_nodes:", num_nodes, "num_edges:", E)

num_nodes: 2077023 num_edges: 31898218


In [7]:

# Build edge_attr from new features
ts = pd.to_datetime(df["_ts"], errors="coerce")

if ts.isna().any():
    ts = ts.fillna(ts.dropna().min())

t0 = ts.min()
t_sec = (ts - t0).dt.total_seconds().astype(np.float32).to_numpy()

t_max = float(t_sec.max()) if float(t_sec.max()) > 0 else 1.0
t_sec = (t_sec / t_max).astype(np.float32)

dow = pd.to_numeric(df["tx_dow"], errors="coerce").fillna(0).astype(np.int64).to_numpy()
dow = np.clip(dow, 0, 6)
dow_oh = np.eye(7, dtype=np.float32)[dow]
is_weekend = pd.to_numeric(df["tx_is_weekend"], errors="coerce").fillna(0).astype(np.float32).to_numpy()
hour_sin = pd.to_numeric(df["tx_hour_sin"], errors="coerce").fillna(0).astype(np.float32).to_numpy()
hour_cos = pd.to_numeric(df["tx_hour_cos"], errors="coerce").fillna(0).astype(np.float32).to_numpy()
amt = pd.to_numeric(df["Log Amount Received"], errors="coerce").fillna(0).astype(np.float32).to_numpy()

edge_attr = np.column_stack([
    amt,
    t_sec,
    hour_sin,
    hour_cos,
    dow_oh,
    is_weekend
]).astype("float32")

y_edge = pd.to_numeric(df["Is Laundering"], errors="coerce").fillna(0).astype("int64").to_numpy()

print("num_nodes:", num_nodes, "num_edges:", E)

num_nodes: 2077023 num_edges: 31898218


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
edge_attr_t = torch.tensor(edge_attr, dtype=torch.float32)
y_edge_t    = torch.tensor(y_edge, dtype=torch.long)

data = Data(num_nodes=num_nodes, edge_index=edge_index, edge_attr=edge_attr_t)

edge_ids = torch.arange(edge_index.size(1), dtype=torch.long)

perm = torch.randperm(edge_ids.numel())
n_train = int(0.80 * perm.numel())
train_eids = edge_ids[perm[:n_train]]
val_eids   = edge_ids[perm[n_train:]]

train_loader = LinkNeighborLoader(
    data,
    edge_label_index=edge_index[:, train_eids],
    edge_label=y_edge_t[train_eids],
    num_neighbors=[15, 10],
    batch_size=4096,
    shuffle=True
)

val_loader = LinkNeighborLoader(
    data,
    edge_label_index=edge_index[:, val_eids],
    edge_label=y_edge_t[val_eids],
    num_neighbors=[15, 10],
    batch_size=4096,
    shuffle=False
)

print("train batches:", len(train_loader), "val batches:", len(val_loader))

EDGE_DIM = data.edge_attr.size(1)
print("EDGE_DIM:", EDGE_DIM)

device: cuda
train batches: 6231 val batches: 1558
EDGE_DIM: 12


In [9]:
class EdgeAwareGNN(nn.Module):
    def __init__(self, num_nodes, hidden=128, edge_dim=EDGE_DIM):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hidden)
        self.conv1 = TransformerConv(hidden, hidden, edge_dim=edge_dim)
        self.conv2 = TransformerConv(hidden, hidden, edge_dim=edge_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, batch):
        x = self.emb(batch.n_id)
        x = F.relu(self.conv1(x, batch.edge_index, batch.edge_attr))
        x = self.conv2(x, batch.edge_index, batch.edge_attr)

        src_i = batch.edge_label_index[0]
        dst_i = batch.edge_label_index[1]
        hs = x[src_i]
        hd = x[dst_i]
        h = torch.cat([hs, hd, hs * hd, (hs - hd).abs()], dim=-1)
        return self.mlp(h)

model = EdgeAwareGNN(num_nodes=num_nodes, hidden=128, edge_dim=EDGE_DIM).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.CrossEntropyLoss()

In [10]:
def run_epoch_train(model, loader, opt, crit, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        logits = model(batch)
        y = batch.edge_label.to(device)

        loss = crit(logits, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        bs = y.numel()
        total_loss += float(loss.item()) * bs

        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(bs)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


@torch.no_grad()
def run_epoch_eval(model, loader, crit, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    probs_all = []
    ys_all = []

    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break

        batch = batch.to(device)
        logits = model(batch)
        y = batch.edge_label.to(device)

        loss = crit(logits, y)

        bs = y.numel()
        total_loss += float(loss.item()) * bs

        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(bs)

        prob1 = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        probs_all.append(prob1)
        ys_all.append(y.detach().cpu().numpy())

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)

    ys = np.concatenate(ys_all) if ys_all else np.array([])
    ps = np.concatenate(probs_all) if probs_all else np.array([])

    if ys.size > 0 and len(np.unique(ys)) > 1:
        pr_auc = average_precision_score(ys, ps)
        roc_auc = roc_auc_score(ys, ps)
    else:
        pr_auc = np.nan
        roc_auc = np.nan

    pos_rate = float(ys.mean()) if ys.size > 0 else np.nan
    return avg_loss, acc, pos_rate, pr_auc, roc_auc

In [11]:
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch_train(model, train_loader, opt, crit, device)
    va_loss, va_acc, _, _, _ = run_epoch_eval(model, val_loader, crit, device, max_batches=200)

    print(
        f"epoch {epoch} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"val loss {va_loss:.4f} acc {va_acc:.4f}"
    )

epoch 1 | train loss 0.0072 acc 0.9989 | val loss 0.0063 acc 0.9990
epoch 2 | train loss 0.0051 acc 0.9991 | val loss 0.0057 acc 0.9991
epoch 3 | train loss 0.0039 acc 0.9992 | val loss 0.0055 acc 0.9991


In [12]:
_, _, pos_rate, pr_auc, roc_auc = run_epoch_eval(model, val_loader, crit, device, max_batches=200)

print()
print("Pos rate:", pos_rate)
print("PR-AUC:", pr_auc)
print("ROC-AUC:", roc_auc)


Pos rate: 0.0011181640625
PR-AUC: 0.33479374444285015
ROC-AUC: 0.9081815403708078


# CatBoost

In [13]:
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip -q install catboost
    from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 26.2 MB/s eta 0:00:00


In [14]:
def encode_batch(model, batch):
    # returns node embeddings for the sampled nodes in this batch
    x = model.emb(batch.n_id)
    x = F.relu(model.conv1(x, batch.edge_index, batch.edge_attr))
    x = model.conv2(x, batch.edge_index, batch.edge_attr)
    return x

hidden = 128  # MUST match your model hidden size

# in-memory embeddings: (num_nodes, hidden)
H = np.zeros((num_nodes, hidden), dtype=np.float32)

infer_loader = NeighborLoader(
    data,
    input_nodes=torch.arange(num_nodes),
    num_neighbors=[10, 5],
    batch_size=65536,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

model.eval()
with torch.no_grad():
    for batch in infer_loader:
        batch = batch.to(device)
        z = encode_batch(model, batch).detach().cpu().numpy()
        H[batch.n_id.cpu().numpy()] = z

print("Embeddings stored in memory:", H.shape, "dtype:", H.dtype)

Embeddings stored in memory: (2077023, 128) dtype: float32


In [15]:
def build_catboost_Xy(edge_ids, edge_index, edge_attr_t, y_edge_t, H):
    if torch.is_tensor(edge_ids):
        edge_ids_np = edge_ids.detach().cpu().numpy()
    else:
        edge_ids_np = edge_ids

    src = edge_index[0, edge_ids_np].detach().cpu().numpy()
    dst = edge_index[1, edge_ids_np].detach().cpu().numpy()

    hs = H[src]
    hd = H[dst]
    diff = np.abs(hs - hd)
    had = hs * hd

    eattr = edge_attr_t[edge_ids_np].detach().cpu().numpy()
    X = np.concatenate([hs, hd, diff, had, eattr], axis=1)

    y = y_edge_t[edge_ids_np].detach().cpu().numpy()
    return X, y

sample_eids = torch.arange(0, min(10000, edge_index.size(1)))
X_tmp, y_tmp = build_catboost_Xy(sample_eids, edge_index, data.edge_attr, y_edge_t, H)
print("CatBoost X shape:", X_tmp.shape, "y pos rate:", float(y_tmp.mean()))

CatBoost X shape: (10000, 524) y pos rate: 0.0


In [16]:
def build_catboost_Xy(edge_ids, edge_index, edge_attr_t, y_edge_t, H):
    if hasattr(edge_ids, "detach"):
        edge_ids_np = edge_ids.detach().cpu().numpy()
    else:
        edge_ids_np = np.asarray(edge_ids)

    src = edge_index[0, edge_ids_np].detach().cpu().numpy()
    dst = edge_index[1, edge_ids_np].detach().cpu().numpy()

    hs = H[src]
    hd = H[dst]
    diff = np.abs(hs - hd)
    had = hs * hd

    eattr = edge_attr_t[edge_ids_np].detach().cpu().numpy()
    X = np.concatenate([hs, hd, diff, had, eattr], axis=1)

    y = y_edge_t[edge_ids_np].detach().cpu().numpy()
    return X, y


In [17]:
ts = pd.to_datetime(df["_ts"], errors="coerce")
if ts.isna().any():
    ts = ts.fillna(ts.dropna().min())

# sort edges by timestamp
order = np.argsort(ts.values.astype("datetime64[ns]"))
E = len(order)
cut = int(0.80 * E)

train_eids = order[:cut]
test_eids  = order[cut:]

y_np = y_edge_t.detach().cpu().numpy()
train_pos = train_eids[y_np[train_eids] == 1]
train_neg = train_eids[y_np[train_eids] == 0]

test_pos_rate = float(y_np[test_eids].mean())
train_pos_rate = float(y_np[train_eids].mean())

print("Train pos rate:", train_pos_rate)
print("Test pos rate:", test_pos_rate)
print("Train edges:", len(train_eids), "Test edges:", len(test_eids))

Train pos rate: 0.0009635726510423348
Test pos rate: 0.0016679614097589144
Train edges: 25518574 Test edges: 6379644


In [18]:
neg_mult = 10
n_pos = len(train_pos)

if n_pos == 0:
    raise ValueError("No positive edges in train split. Something is wrong with labels or split.")

n_neg = min(len(train_neg), n_pos * neg_mult)

rng = np.random.default_rng(42)
train_neg_sample = rng.choice(train_neg, size=n_neg, replace=False)

train_sel = np.concatenate([train_pos, train_neg_sample])
rng.shuffle(train_sel)

print("CatBoost train_sel:", len(train_sel), "pos:", n_pos, "neg:", n_neg, "pos_rate:", n_pos / len(train_sel))

CatBoost train_sel: 270479 pos: 24589 neg: 245890 pos_rate: 0.09090909090909091


In [19]:
X_train, y_train = build_catboost_Xy(train_sel, edge_index, data.edge_attr, y_edge_t, H)
X_test, y_test = build_catboost_Xy(test_eids, edge_index, data.edge_attr, y_edge_t, H)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (270479, 524) X_test: (6379644, 524)


In [20]:
cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    task_type="GPU" if False else "CPU"
)

cb.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.9461996	best: 0.9461996 (0)	total: 603ms	remaining: 20m 6s
200:	test: 0.9790037	best: 0.9790037 (200)	total: 1m 45s	remaining: 15m 45s
400:	test: 0.9808811	best: 0.9808811 (400)	total: 3m 28s	remaining: 13m 51s
600:	test: 0.9814973	best: 0.9814973 (600)	total: 5m 10s	remaining: 12m 1s
800:	test: 0.9817543	best: 0.9817566 (798)	total: 6m 49s	remaining: 10m 12s
1000:	test: 0.9819189	best: 0.9819348 (972)	total: 8m 29s	remaining: 8m 28s
1200:	test: 0.9820119	best: 0.9820232 (1195)	total: 10m 8s	remaining: 6m 44s
1400:	test: 0.9820474	best: 0.9820511 (1392)	total: 11m 46s	remaining: 5m 2s
1600:	test: 0.9820684	best: 0.9820684 (1600)	total: 13m 25s	remaining: 3m 20s
1800:	test: 0.9821130	best: 0.9821143 (1798)	total: 15m 4s	remaining: 1m 39s
1999:	test: 0.9821033	best: 0.9821398 (1893)	total: 16m 43s	remaining: 0us

bestTest = 0.9821397805
bestIteration = 1893

Shrink model to first 1894 iterations.


CatBoostClassifier(depth=8, eval_metric='AUC', iterations=2000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='CPU', verbose=200)

In [21]:
# Evaluate CatBoost

p_test = cb.predict_proba(X_test)[:, 1]

print()
print("Pos rate:", float(y_test.mean()))
print("PR-AUC:", average_precision_score(y_test, p_test))
print("ROC-AUC:", roc_auc_score(y_test, p_test))


Pos rate: 0.0016679614097589144
PR-AUC: 0.6385117987891411
ROC-AUC: 0.982139780480256
